# 00 — Data audit (no model training)

**Goal:** understand the AnnData returned by
`scvi.data.heart_cell_atlas_subsampled`, verify where the raw counts live, and
fix the `donor_key`, `batch_key` and `cell_type_key` from evidence rather than
column names.

**Leakage risk here:** none — this notebook only inspects data. It does not
split or train anything.

**Stop condition:** if `X` does not look like non-negative integer UMI
counts, do not continue to the modelling notebooks.

In [ ]:
import sys
from pathlib import Path
REPO = Path.cwd()
if (REPO / "src").exists():
    sys.path.insert(0, str(REPO / "src"))
import heartmap
print("heartmap from:", Path(heartmap.__file__).parent)


## 1. Load the official dataset

The loader downloads the ~20k-cell subsampled Human Heart Cell Atlas and, with `remove_nuisance_clusters=True`, drops doublet/NotAssigned clusters.

In [ ]:
import scvi
from heartmap.config import load_config

cfg = load_config("configs/main.yaml")
adata = scvi.data.heart_cell_atlas_subsampled(
    save_path=str(cfg.data_raw_dir),
    remove_nuisance_clusters=True,
)
print(adata)


## 2. AnnData structure: X, layers, raw, obsm, varm, uns

We need to know whether counts are in `X`, a layer, or `.raw` before choosing `layer="counts"` for scVI.

In [ ]:
print("X:", adata.X.__class__.__name__, adata.X.shape, adata.X.dtype)
print("layers:", dict(adata.layers))
print("raw:", adata.raw)
print("obsm keys:", list(adata.obsm.keys()))
print("varm keys:", list(adata.varm.keys()))
print("uns keys:", list(adata.uns.keys()))


## 3. Are the values raw counts?

Non-negative, integer-valued non-zeros with a count-like range are expected. The package validator raises if this is not the case.

In [ ]:
import numpy as np
from heartmap.data import validate_counts

info = validate_counts(adata, None)  # inspects X
for k, v in info.items():
    print(f"{k}: {v}")
assert info["looks_like_integer_counts"], "STOP: X does not look like raw counts"


## 4. Expose the audited counts layer

Every downstream step refers explicitly to `layers['counts']`.

In [ ]:
adata.layers[cfg["counts_layer"]] = adata.X.copy()
validate_counts(adata, cfg["counts_layer"])


## 5. Inspect every `obs` column

Candidate donor/batch/label columns must be identified by their contents, not their names.

In [ ]:
import pandas as pd
rows = []
for col in adata.obs.columns:
    s = adata.obs[col]
    rows.append(dict(
        column=col, dtype=str(s.dtype),
        n_unique=s.nunique(),
        n_missing=int(s.isna().sum()),
        example=str(s.iloc[0])[:40],
    ))
pd.DataFrame(rows)


## 6. Donor, batch and cell-type candidates

In [ ]:
for col in ["donor", "sample", "source", "region", "cell_source",
            "cell_type", "cell_states", "type"]:
    if col in adata.obs.columns:
        vc = adata.obs[col].value_counts()
        print(f"\n=== {col} ({len(vc)} unique) ===")
        print(vc.head(15))


## 7. Donor × cell-type cross-tabulation

This table later drives the deterministic query-donor eligibility rule (≥ 500 cells, ≥ 5 broad types, major types shared with other donors).

In [ ]:
ctab = pd.crosstab(adata.obs[cfg["donor_key"]],
                   adata.obs[cfg["cell_type_key"]])
ctab


## 8. Decision (fixed in `configs/main.yaml`)

- `donor_key = 'donor'` — 14 heart donors; `sample` (145 values) is a within-donor unit and would leak the donor if used as batch.
- `batch_key = 'donor'` — the held-out biological/technical unit.
- `cell_type_key = 'cell_type'` — 11 author-provided **broad** types. `cell_states` (65 fine states) is future work and is removed from query inputs.
- `counts_layer = 'counts'` — copy of the verified raw-count `X`.

The full audit is also produced by `scripts/audit_data.py` and stored in `results/data_audit.json` and `docs/DATA_DICTIONARY.md`.